# Model 2 Training: Convolutional Autoencoder for Image Restoration (50 Epochs)

This notebook trains Model 2: Deep Convolutional Autoencoder (Encoder-Decoder architecture with spatial compression, latent bottleneck, and skip connections) on Kaggle Dual GPUs for 50 epochs.

### Accelerator Configuration: Tesla T4 Dual GPU (GPU T4 x 2)
- Multi-GPU DataParallel Support: Automatically detects and distributes batches across GPU 0 and GPU 1.
- Memory-Optimized Training Pipeline: Implements periodic CUDA cache cleanup, persistent DataLoader workers, and Python garbage collection to prevent silent kernel termination / OOM crashes during long 50-epoch runs.

### Key Implementation Details:
- Architecture: Convolutional Autoencoder (Encoder-Decoder with spatial downsampling, bottleneck representation, and skip connections).
- Multi-GPU Execution: Wrapped with torch.nn.DataParallel across multiple GPUs.
- Cell Execution Timers: Tracks individual cell execution time and cumulative elapsed time.
- Training Duration: Configured for 50 full epochs with Cosine Annealing Learning Rate scheduling.
- Model Checkpointing: Saves the best performing model based on Validation PSNR to 'autoencoder_model.pth'.
- Results Export: Packages autoencoder_model.pth, training_metrics.json, training curves, and test sample visualizations into autoencoder_restoration_results.zip.

In [ ]:
import time
GLOBAL_START_TIME = time.time()
_cell_start = time.time()

import os
import gc
import glob
import json
import math
import random
import zipfile
from pathlib import Path
from PIL import Image
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def print_timer(cell_name, cell_start):
    cell_elapsed = time.time() - cell_start
    total_elapsed = time.time() - GLOBAL_START_TIME
    print(f"[Cell Timer - {cell_name}] Cell Time: {cell_elapsed:.2f}s | Cumulative Time: {total_elapsed:.2f}s")

print_timer("Imports and Setup", _cell_start)

In [ ]:
_cell_start = time.time()

# Environment and Dataset Path Resolution
PRIMARY_DATASET_PATH = "/kaggle/input/datasets/hetulkadiya/degraded-dataset-zip-file/degraded_dataset"

def resolve_dataset_path():
    if os.path.exists(PRIMARY_DATASET_PATH):
        return PRIMARY_DATASET_PATH
    matches = glob.glob("/kaggle/input/**/degraded_dataset", recursive=True)
    if matches:
        return matches[0]
    if os.path.exists("../data/degraded_dataset"):
        return "../data/degraded_dataset"
    if os.path.exists("./degraded_dataset"):
        return "./degraded_dataset"
    return PRIMARY_DATASET_PATH

DATASET_ROOT = resolve_dataset_path()
OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."

# Multi-GPU Accelerator Setup
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

def get_compute_device():
    if NUM_GPUS > 0:
        try:
            for i in range(NUM_GPUS):
                gpu_name = torch.cuda.get_device_name(i)
                print(f"Detected GPU {i}: {gpu_name}")
            
            test_tensor = torch.randn(2, 3, 16, 16, device="cuda:0")
            test_conv = nn.Conv2d(3, 16, 3, padding=1).to("cuda:0")
            _ = test_conv(test_tensor)
            return torch.device("cuda")
        except Exception as e:
            print(f"Warning: CUDA error encountered on current GPU ({e}).")
            print("Recommendation: In Kaggle Notebook Settings (right sidebar), switch Accelerator to 'GPU T4 x 2'.")
            return torch.device("cpu")
    return torch.device("cpu")

DEVICE = get_compute_device()
print(f"Active Main Device: {DEVICE} (Total GPUs Available: {NUM_GPUS})")
print(f"Dataset Root: {DATASET_ROOT}")
print(f"Output Directory: {OUTPUT_DIR}")

print_timer("Path and Multi-GPU Device Setup", _cell_start)

In [ ]:
_cell_start = time.time()

# Dataset Definition for 7-Combination Paired Degraded Images
class KaggleDegradedDataset(Dataset):
    def __init__(self, root_dir, split="train", target_size=(128, 128)):
        self.root_dir = root_dir
        self.split = split
        self.target_size = target_size
        
        self.split_dir = os.path.join(root_dir, split)
        self.clean_dir = os.path.join(self.split_dir, "clean")
        
        self.combos = [
            "c1_noise_lowlight",
            "c2_noise_blur",
            "c3_lowlight_blur",
            "c4_noise_only",
            "c5_lowlight_only",
            "c6_blur_only",
            "c7_all_three"
        ]
        
        self.pairs = []
        if os.path.exists(self.clean_dir):
            clean_files = sorted(glob.glob(os.path.join(self.clean_dir, "*.png")) + glob.glob(os.path.join(self.clean_dir, "*.jpg")))
            for clean_path in clean_files:
                filename = os.path.basename(clean_path)
                for combo in self.combos:
                    deg_path = os.path.join(self.split_dir, combo, filename)
                    if os.path.exists(deg_path):
                        self.pairs.append((deg_path, clean_path, combo))
                        
        self.to_tensor = T.Compose([
            T.Resize(target_size),
            T.ToTensor()
        ])
        print(f"Loaded [{split.upper()}] dataset split: {len(self.pairs)} paired samples.")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        deg_path, clean_path, combo = self.pairs[idx]
        with Image.open(deg_path) as deg_img, Image.open(clean_path) as clean_img:
            deg_tensor = self.to_tensor(deg_img.convert("RGB"))
            clean_tensor = self.to_tensor(clean_img.convert("RGB"))
        return deg_tensor, clean_tensor

# Create Datasets and Memory-Safe DataLoaders
USE_CUDA = (DEVICE.type == "cuda")
# Batch size 64 for Dual T4 GPUs (32 per GPU) or 32 for Single GPU
BATCH_SIZE = 64 if NUM_GPUS > 1 else (32 if USE_CUDA else 8)
# 2 workers per GPU with persistent workers prevents worker recreation memory leaks
NUM_WORKERS = (2 * NUM_GPUS) if USE_CUDA else 0

train_dataset = KaggleDegradedDataset(DATASET_ROOT, split="train")
val_dataset = KaggleDegradedDataset(DATASET_ROOT, split="val")
test_dataset = KaggleDegradedDataset(DATASET_ROOT, split="test")

loader_kwargs = {
    "num_workers": NUM_WORKERS,
    "pin_memory": USE_CUDA,
    "persistent_workers": (NUM_WORKERS > 0)
}

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, **loader_kwargs)

print(f"DataLoader Configured: Batch Size = {BATCH_SIZE} | Workers = {NUM_WORKERS} | Persistent Workers = {NUM_WORKERS > 0}")
print_timer("Dataset and DataLoaders Setup", _cell_start)

In [ ]:
_cell_start = time.time()

# Model 2: Convolutional Autoencoder Architecture (Encoder-Decoder with Latent Bottleneck)
class ConvolutionalAutoencoder(nn.Module):
    """
    Model 2: Convolutional Autoencoder.
    Encodes degraded image into a compressed latent bottleneck representation
    and decodes back to high-resolution clean image with spatial skip connections.
    """
    def __init__(self, in_channels=3, out_channels=3, base_features=64):
        super(ConvolutionalAutoencoder, self).__init__()
        
        # Encoder Blocks
        self.enc1 = nn.Sequential(
            nn.Conv2d(in_channels, base_features, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_features),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_features, base_features, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_features),
            nn.ReLU(inplace=True)
        )
        self.pool1 = nn.MaxPool2d(2, 2)  # 128 -> 64
        
        self.enc2 = nn.Sequential(
            nn.Conv2d(base_features, base_features * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_features * 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_features * 2, base_features * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_features * 2),
            nn.ReLU(inplace=True)
        )
        self.pool2 = nn.MaxPool2d(2, 2)  # 64 -> 32
        
        # Latent Bottleneck Block
        self.bottleneck = nn.Sequential(
            nn.Conv2d(base_features * 2, base_features * 4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_features * 4),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_features * 4, base_features * 4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_features * 4),
            nn.ReLU(inplace=True)
        )
        
        # Decoder Blocks with Transposed Convolutions
        self.up2 = nn.ConvTranspose2d(base_features * 4, base_features * 2, kernel_size=2, stride=2) # 32 -> 64
        self.dec2 = nn.Sequential(
            nn.Conv2d(base_features * 4, base_features * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_features * 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_features * 2, base_features * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_features * 2),
            nn.ReLU(inplace=True)
        )
        
        self.up1 = nn.ConvTranspose2d(base_features * 2, base_features, kernel_size=2, stride=2) # 64 -> 128
        self.dec1 = nn.Sequential(
            nn.Conv2d(base_features * 2, base_features, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_features),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_features, base_features, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_features),
            nn.ReLU(inplace=True)
        )
        
        self.final = nn.Sequential(
            nn.Conv2d(base_features, out_channels, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # Encoding
        e1 = self.enc1(x)
        p1 = self.pool1(e1)
        
        e2 = self.enc2(p1)
        p2 = self.pool2(e2)
        
        # Latent Bottleneck
        b = self.bottleneck(p2)
        
        # Decoding with Skip Connections
        d2 = self.up2(b)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)
        
        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)
        
        out = self.final(d1)
        return out

base_model = ConvolutionalAutoencoder()

# Wrap model with DataParallel if multiple GPUs are detected
if NUM_GPUS > 1:
    print(f"Enabling Multi-GPU DataParallel across {NUM_GPUS} GPUs!")
    model = nn.DataParallel(base_model)
else:
    model = base_model

model = model.to(DEVICE)

num_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
print(f"Convolutional Autoencoder Restoration Model initialized. Total Trainable Parameters: {num_params:,}")

print_timer("Multi-GPU Autoencoder Model Setup", _cell_start)

In [ ]:
_cell_start = time.time()

# Loss Function and Metric Evaluation Functions (PSNR and SSIM)
class CombinedLoss(nn.Module):
    def __init__(self, l1_weight=0.8, mse_weight=0.2):
        super(CombinedLoss, self).__init__()
        self.l1_weight = l1_weight
        self.mse_weight = mse_weight
        self.l1 = nn.L1Loss()
        self.mse = nn.MSELoss()

    def forward(self, pred, target):
        return self.l1_weight * self.l1(pred, target) + self.mse_weight * self.mse(pred, target)

def calculate_batch_psnr(pred, target):
    with torch.no_grad():
        mse = torch.mean((pred - target) ** 2, dim=[1, 2, 3])
        psnr = 10.0 * torch.log10(1.0 / (mse + 1e-8))
        return torch.mean(psnr).item()

def calculate_batch_ssim(pred, target):
    with torch.no_grad():
        mu_x = F.avg_pool2d(pred, 3, 1, 1)
        mu_y = F.avg_pool2d(target, 3, 1, 1)
        sigma_x = F.avg_pool2d(pred * pred, 3, 1, 1) - mu_x ** 2
        sigma_y = F.avg_pool2d(target * target, 3, 1, 1) - mu_y ** 2
        sigma_xy = F.avg_pool2d(pred * target, 3, 1, 1) - mu_x * mu_y
        C1, C2 = 0.01 ** 2, 0.03 ** 2
        ssim_map = ((2 * mu_x * mu_y + C1) * (2 * sigma_xy + C2)) / ((mu_x ** 2 + mu_y ** 2 + C1) * (sigma_x + sigma_y + C2))
        return torch.mean(ssim_map).item()

def extract_clean_state_dict(m):
    return m.module.state_dict() if isinstance(m, nn.DataParallel) else m.state_dict()

def load_clean_state_dict(m, path, device):
    state_dict = torch.load(path, map_location=device)
    if isinstance(m, nn.DataParallel):
        m.module.load_state_dict(state_dict)
    else:
        m.load_state_dict(state_dict)

criterion = CombinedLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, betas=(0.9, 0.999), weight_decay=1e-5)

# Learning Rate Scheduler for 50 Epochs
EPOCHS = 50
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

print_timer("Loss and Metric Functions Setup", _cell_start)

In [ ]:
_cell_start = time.time()

# 50-Epoch Training and Validation Loop with Memory Leak Prevention
MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, "autoencoder_model.pth")
best_val_psnr = 0.0

history = {
    "train_loss": [],
    "val_loss": [],
    "val_psnr": [],
    "val_ssim": [],
    "lr": []
}

print(f"Starting Multi-GPU Training for {EPOCHS} Epochs on {DEVICE}...")

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    
    # Training Phase
    model.train()
    train_loss = 0.0
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS:02d} [Train]", leave=False, mininterval=1.0)
    
    for degraded, clean in train_pbar:
        degraded, clean = degraded.to(DEVICE, non_blocking=True), clean.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        output = model(degraded)
        loss = criterion(output, clean)
        loss.backward()
        optimizer.step()
        
        batch_loss = loss.item()
        train_loss += batch_loss
        train_pbar.set_postfix({"Loss": f"{batch_loss:.4f}"})
        
    train_loss /= max(len(train_loader), 1)
    current_lr = optimizer.param_groups[0]["lr"]
    history["train_loss"].append(round(train_loss, 4))
    history["lr"].append(round(current_lr, 8))
    
    # Validation Phase
    model.eval()
    val_loss = 0.0
    val_psnr = 0.0
    val_ssim = 0.0
    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch:02d}/{EPOCHS:02d} [Val]  ", leave=False, mininterval=1.0)
    
    with torch.no_grad():
        for degraded, clean in val_pbar:
            degraded, clean = degraded.to(DEVICE, non_blocking=True), clean.to(DEVICE, non_blocking=True)
            restored = model(degraded)
            loss = criterion(restored, clean)
            val_loss += loss.item()
            val_psnr += calculate_batch_psnr(restored, clean)
            val_ssim += calculate_batch_ssim(restored, clean)
            
    val_loss /= max(len(val_loader), 1)
    val_psnr /= max(len(val_loader), 1)
    val_ssim /= max(len(val_loader), 1)
    
    history["val_loss"].append(round(val_loss, 4))
    history["val_psnr"].append(round(val_psnr, 2))
    history["val_ssim"].append(round(val_ssim, 4))
    
    # Step the learning rate scheduler
    scheduler.step()
    
    epoch_time = time.time() - epoch_start
    print(f"Epoch {epoch:02d}/{EPOCHS:02d} [{epoch_time:.1f}s] -> Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val PSNR: {val_psnr:.2f} dB | Val SSIM: {val_ssim:.4f} | LR: {current_lr:.2e}")
    
    if val_psnr > best_val_psnr:
        best_val_psnr = val_psnr
        clean_sd = extract_clean_state_dict(model)
        torch.save(clean_sd, MODEL_SAVE_PATH)
        print(f"  --> Saved best checkpoint to '{MODEL_SAVE_PATH}' (Val PSNR: {val_psnr:.2f} dB)")
        
    # Memory Management: Clear GPU cache and garbage collect every epoch to prevent memory leaks
    if USE_CUDA:
        torch.cuda.empty_cache()
    gc.collect()

print_timer("50-Epoch Multi-GPU Training Loop", _cell_start)

In [ ]:
_cell_start = time.time()

# Final Evaluation and Benchmark on Test Set
if os.path.exists(MODEL_SAVE_PATH):
    load_clean_state_dict(model, MODEL_SAVE_PATH, DEVICE)

model.eval()
deg_psnr_total, deg_ssim_total = 0.0, 0.0
res_psnr_total, res_ssim_total = 0.0, 0.0

with torch.no_grad():
    for degraded, clean in tqdm(test_loader, desc="Evaluating Test Set", mininterval=1.0):
        degraded, clean = degraded.to(DEVICE), clean.to(DEVICE)
        restored = model(degraded)
        
        deg_psnr_total += calculate_batch_psnr(degraded, clean)
        deg_ssim_total += calculate_batch_ssim(degraded, clean)
        
        res_psnr_total += calculate_batch_psnr(restored, clean)
        res_ssim_total += calculate_batch_ssim(restored, clean)

n_test_batches = max(len(test_loader), 1)
avg_deg_psnr = deg_psnr_total / n_test_batches
avg_deg_ssim = deg_ssim_total / n_test_batches
avg_res_psnr = res_psnr_total / n_test_batches
avg_res_ssim = res_ssim_total / n_test_batches

psnr_gain = avg_res_psnr - avg_deg_psnr
ssim_gain = avg_res_ssim - avg_deg_ssim

print("\n======================================================")
print("MODEL 2 (CONVOLUTIONAL AUTOENCODER - 50 EPOCHS) TEST BENCHMARK")
print("======================================================")
print(f"GPUs Utilized           : {max(NUM_GPUS, 1)}")
print(f"Degraded Input Quality  : PSNR = {avg_deg_psnr:.2f} dB | SSIM = {avg_deg_ssim:.4f}")
print(f"Autoencoder Restored    : PSNR = {avg_res_psnr:.2f} dB | SSIM = {avg_res_ssim:.4f}")
print(f"Restoration dB Gain     : PSNR Gain = +{psnr_gain:.2f} dB | SSIM Gain = +{ssim_gain:.4f}")
print("======================================================")

# Save Evaluation Metrics JSON
metrics_json_path = os.path.join(OUTPUT_DIR, "training_metrics.json")
metrics_data = {
    "model_name": "Convolutional Autoencoder Restoration Network (50 Epochs)",
    "gpus_used": max(NUM_GPUS, 1),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "loss_functions": ["L1Loss", "MSELoss"],
    "optimizer": "Adam (lr=1e-4, CosineAnnealing)",
    "history": history,
    "test_results": {
        "degraded_psnr": round(avg_deg_psnr, 2),
        "degraded_ssim": round(avg_deg_ssim, 4),
        "restored_psnr": round(avg_res_psnr, 2),
        "restored_ssim": round(avg_res_ssim, 4),
        "psnr_gain_db": round(psnr_gain, 2),
        "ssim_gain": round(ssim_gain, 4)
    }
}
with open(metrics_json_path, "w") as f:
    json.dump(metrics_data, f, indent=2)

print_timer("Test Evaluation and Metrics Export", _cell_start)

In [ ]:
_cell_start = time.time()

# Plot Training Curves and Generate Sample Visual Restorations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss Curve
axes[0].plot(range(1, EPOCHS + 1), history["train_loss"], label="Train Loss", marker="o", markersize=3, color="#ef4444")
axes[0].plot(range(1, EPOCHS + 1), history["val_loss"], label="Val Loss", marker="s", markersize=3, color="#3b82f6")
axes[0].set_title("L1 + MSE Loss Curve (50 Epochs)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, linestyle="--", alpha=0.5)
axes[0].legend()

# PSNR Curve
axes[1].plot(range(1, EPOCHS + 1), history["val_psnr"], label="Val PSNR (dB)", marker="^", markersize=3, color="#10b981")
axes[1].set_title("Validation PSNR Curve (dB)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("PSNR (dB)")
axes[1].grid(True, linestyle="--", alpha=0.5)
axes[1].legend()

# SSIM Curve
axes[2].plot(range(1, EPOCHS + 1), history["val_ssim"], label="Val SSIM", marker="d", markersize=3, color="#8b5cf6")
axes[2].set_title("Validation SSIM Score Curve", fontsize=12, fontweight="bold")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("SSIM Score")
axes[2].grid(True, linestyle="--", alpha=0.5)
axes[2].legend()

plt.tight_layout()
plots_image_path = os.path.join(OUTPUT_DIR, "loss_psnr_ssim_curves.png")
plt.savefig(plots_image_path, dpi=300)
plt.show()

# Visual Restoration Samples Plot
model.eval()
for deg_b, clean_b in test_loader:
    deg_b, clean_b = deg_b.to(DEVICE), clean_b.to(DEVICE)
    with torch.no_grad():
        res_b = model(deg_b)
    break

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for i in range(min(3, deg_b.shape[0])):
    deg_np = deg_b[i].cpu().permute(1, 2, 0).numpy()
    res_np = res_b[i].cpu().permute(1, 2, 0).numpy()
    clean_np = clean_b[i].cpu().permute(1, 2, 0).numpy()
    
    axes[i, 0].imshow(clean_np)
    axes[i, 0].set_title("Original Clean", fontsize=9)
    axes[i, 0].axis("off")
    
    axes[i, 1].imshow(deg_np)
    axes[i, 1].set_title("Degraded Input", fontsize=9)
    axes[i, 1].axis("off")
    
    axes[i, 2].imshow(res_np)
    axes[i, 2].set_title("Autoencoder Restored", fontsize=9)
    axes[i, 2].axis("off")

plt.suptitle("Convolutional Autoencoder Test Sample Restorations (50 Epochs)", fontsize=13)
plt.tight_layout()
samples_image_path = os.path.join(OUTPUT_DIR, "test_restoration_samples.png")
plt.savefig(samples_image_path, dpi=300)
plt.show()

print_timer("Plots and Visual Restorations", _cell_start)

In [ ]:
_cell_start = time.time()

# Package Checkpoints, Metrics, Plots into autoencoder_restoration_results.zip
ZIP_FILE_PATH = os.path.join(OUTPUT_DIR, "autoencoder_restoration_results.zip")

files_to_zip = [
    (MODEL_SAVE_PATH, "autoencoder_model.pth"),
    (metrics_json_path, "training_metrics.json"),
    (plots_image_path, "loss_psnr_ssim_curves.png"),
    (samples_image_path, "test_restoration_samples.png")
]

with zipfile.ZipFile(ZIP_FILE_PATH, "w", zipfile.ZIP_DEFLATED) as zipf:
    for src_path, arcname in files_to_zip:
        if os.path.exists(src_path):
            zipf.write(src_path, arcname=arcname)
            print(f"Added to Zip: {arcname}")

print("\n======================================================")
print("RESULTS ZIP ARCHIVE CREATED SUCCESSFULLY")
print(f"Zip Path: {ZIP_FILE_PATH}")
print(f"Size: {os.path.getsize(ZIP_FILE_PATH) / (1024 * 1024):.2f} MB")
print("Contains: autoencoder_model.pth, training_metrics.json, curves, visual samples")
print("======================================================")

print_timer("Zip Packaging and Export", _cell_start)